# 03 — AI Data Cleaning

**Author:** Ricardo Sanchez  

**Project:** Monetary Policy Shocks and Startup Funding Transitions (Seed → Series A)

**Date:** March 03, 2026 

**Data Source:** Crunchbase (2016 – 2023 Q3)
  
**Sector:** Artificial Intelligence

---

## Objective

Transform raw Crunchbase AI funding data into a clean, analysis-ready survival dataset that tracks each startup's time from Seed funding to Series A. This notebook mirrors the structure of `02_fintech_data_cleaning.ipynb` to ensure consistent treatment across sectors. The output is a single CSV with one row per firm.

## Pipeline Overview

| Stage | Description |
|-------|-------------|
| **Part 1** | Merge raw seed and Series A funding packages into single files |
| **Part 2** | Build the survival dataset — collapse to one round per firm, define event/censoring, compute survival times |
| **Part 3** | Assign metro regions, attach covariates, label sector |
| **Part 4** | Select final columns and export |

## Inputs
- `data/raw/AI/ai_seed_package_A.csv`
- `data/raw/AI/ai_seed_package_B.csv`
- `data/raw/AI/ai_series_A_funding_package_A.csv`
- `data/raw/AI/ai_series_A_funding_package_B.xlsx`
- `data/cleaned/ai_loc_map/ai_city_to_metro.csv`

## Outputs
- `data/cleaned/cleaned_ai/ai_simple_survival.csv`

---

## Part 1 — Merge Raw Funding Packages

### 1.1 Setup & Load Raw Seed Packages

Crunchbase seed funding data was downloaded in two batches (package A and package B). Both are CSV files.

In [1]:
# import necessary libraries
import pandas as pd
from datetime import datetime

In [2]:
# Load raw Crunchbase AI seed funding exports (two download batches)
seed_data_1 = pd.read_csv(r"..\data\raw\AI\ai_seed_package_A.csv")
seed_data_2 = pd.read_csv(r"..\data\raw\AI\ai_seed_package_B.csv")

In [3]:
# Inspect date boundaries of each package before merging
print(f'Seed packet A first 3\n{seed_data_1['Announced Date'].head(3)}')
print(f'Seed packet A last 3\n{seed_data_1['Announced Date'].tail(3)} \n')
print(f'Seed packet B first 3\n{seed_data_2['Announced Date'].head(3)}')
print(f'Seed packet B last 3\n{seed_data_2['Announced Date'].tail(3)}')

Seed packet A first 3
0    2016-01-01
1    2016-01-01
2    2016-01-01
Name: Announced Date, dtype: object
Seed packet A last 3
997    2022-01-01
998    2022-01-01
999    2022-01-01
Name: Announced Date, dtype: object 

Seed packet B first 3
0    2024-12-19
1    2024-12-11
2    2024-12-11
Name: Announced Date, dtype: object
Seed packet B last 3
529    2022-01-01
530    2022-01-01
531    2022-01-01
Name: Announced Date, dtype: object


In [4]:
# Sort both packages by announcement date in ascending order for clean concatenation
seed_data_1 = seed_data_1.sort_values(by="Announced Date", ascending=True)
seed_data_2 = seed_data_2.sort_values(by="Announced Date", ascending=True)

print(f'Seed packet A first 3\n{seed_data_1['Announced Date'].head(3)}')
print(f'Seed packet A last 3\n{seed_data_1['Announced Date'].tail(3)} \n')
print(f'Seed packet B first 3\n{seed_data_2['Announced Date'].head(3)}')
print(f'Seed packet B last 3\n{seed_data_2['Announced Date'].tail(3)}')

Seed packet A first 3
0    2016-01-01
1    2016-01-01
2    2016-01-01
Name: Announced Date, dtype: object
Seed packet A last 3
998    2022-01-01
996    2022-01-01
999    2022-01-01
Name: Announced Date, dtype: object 

Seed packet B first 3
531    2022-01-01
530    2022-01-01
529    2022-01-01
Name: Announced Date, dtype: object
Seed packet B last 3
2    2024-12-11
1    2024-12-11
0    2024-12-19
Name: Announced Date, dtype: object


In [5]:
# Concatenate the two sorted packages into a single seed funding dataframe
combined_seed_data = pd.concat([seed_data_1, seed_data_2], ignore_index=True)

print(f'Merged data first 3\n{combined_seed_data['Announced Date'].head(3)} \n')
print(f'Merged data last 3\n{combined_seed_data['Announced Date'].tail(3)}')

Merged data first 3
0    2016-01-01
1    2016-01-01
2    2016-01-01
Name: Announced Date, dtype: object 

Merged data last 3
1529    2024-12-11
1530    2024-12-11
1531    2024-12-19
Name: Announced Date, dtype: object


In [6]:
# Export merged seed data for downstream consumption
combined_seed_data.to_csv(r"..\data\raw\AI\ai_seed_funding_merged.csv", index=False)

### 1.2 Merge Series A Packages

The AI Series A data was also downloaded in two batches. Note that package B is an **Excel file** (`.xlsx`), not CSV — `pd.read_excel` handles this automatically.

In [7]:
# Load Series A packages — note: package B is .xlsx format
series_a_data_1 = pd.read_csv(r"..\data\raw\AI\ai_series_A_funding_package_A.csv")
series_a_data_2 = pd.read_excel(r"..\data\raw\AI\ai_series_A_funding_package_B.xlsx")

# Parse dates immediately to ensure consistent format across both packages
series_a_data_1["Announced Date"] = pd.to_datetime(series_a_data_1["Announced Date"], errors='coerce')
series_a_data_2["Announced Date"] = pd.to_datetime(series_a_data_2["Announced Date"], errors='coerce')

In [8]:
# Inspect date boundaries of each Series A package
print(f'Series A packet A first 3\n{series_a_data_1['Announced Date'].head(3)}')
print(f'Series A packet A last 3\n{series_a_data_1['Announced Date'].tail(3)} \n')
print(f'Series A packet B first 3\n{series_a_data_2['Announced Date'].head(3)}')
print(f'Series A packet B last 3\n{series_a_data_2['Announced Date'].tail(3)}')

Series A packet A first 3
0   2016-01-05
1   2016-01-21
2   2016-03-16
Name: Announced Date, dtype: datetime64[ns]
Series A packet A last 3
465   2022-04-08
466   2022-04-11
467   2022-04-15
Name: Announced Date, dtype: datetime64[ns] 

Series A packet B first 3
0   2022-02-22
1   2022-02-28
2   2022-03-01
Name: Announced Date, dtype: datetime64[ns]
Series A packet B last 3
212   2024-12-19
213   2024-12-21
214   2024-12-23
Name: Announced Date, dtype: datetime64[ns]


In [9]:
# Concatenate both Series A packages into a single dataframe
combined_series_a_data = pd.concat([series_a_data_1, series_a_data_2], ignore_index=True)

print(f'Merged data first 3\n{combined_series_a_data['Announced Date'].head(3)} \n')
print(f'Merged data last 3\n{combined_series_a_data['Announced Date'].tail(3)}')

Merged data first 3
0   2016-01-05
1   2016-01-21
2   2016-03-16
Name: Announced Date, dtype: datetime64[ns] 

Merged data last 3
680   2024-12-19
681   2024-12-21
682   2024-12-23
Name: Announced Date, dtype: datetime64[ns]


In [10]:
# Export merged Series A data for downstream consumption
combined_series_a_data.to_csv(r"..\data\raw\AI\ai_series_A_funding_merged.csv", index=False)

---

## Part 2 — Build Survival Dataset

Construct a simple survival dataset suitable for Kaplan-Meier estimation and downstream hazard modeling. Each row represents one AI startup, tracking the time from its earliest Seed round to its earliest Series A round (or right-censoring if no Series A was observed).

### 2.0 Load Merged Data

In [11]:
# load the merged seed and Series A datasets
ai_seed = pd.read_csv(r"..\data\raw\AI\ai_seed_funding_merged.csv")
ai_series_a = pd.read_csv(r"..\data\raw\AI\ai_series_A_funding_merged.csv")

In [12]:
# Non-null counts per column — seed data
ai_seed.count()

Transaction Name               1532
Transaction Name URL           1532
Organization Location          1532
Organization Name              1532
Organization Name URL          1532
Funding Type                   1532
Money Raised                   1532
Money Raised Currency          1532
Money Raised (in USD)          1532
Announced Date                 1532
Diversity Spotlight             393
Number of Partner Investors     623
Number of Investors            1231
Investor Names                 1231
dtype: int64

In [13]:
# Non-null counts per column — Series A data
ai_series_a.count()

Transaction Name               683
Transaction Name URL           468
Organization Location          683
Organization Name              683
Organization Name URL          468
Funding Type                   683
Money Raised                   683
Money Raised Currency          468
Money Raised (in USD)          468
Announced Date                 683
Diversity Spotlight            328
Number of Partner Investors    540
Number of Investors            675
Investor Names                 675
dtype: int64

In [14]:
# Filter to correct funding round types only (guard against Crunchbase export contamination)
ai_seed = ai_seed[ai_seed["Funding Type"] == "Seed"].copy()
ai_series_a = ai_series_a[ai_series_a["Funding Type"] == "Series A"].copy()

In [15]:
# Verify row counts after funding-type filter — seed
ai_seed.count()

Transaction Name               1532
Transaction Name URL           1532
Organization Location          1532
Organization Name              1532
Organization Name URL          1532
Funding Type                   1532
Money Raised                   1532
Money Raised Currency          1532
Money Raised (in USD)          1532
Announced Date                 1532
Diversity Spotlight             393
Number of Partner Investors     623
Number of Investors            1231
Investor Names                 1231
dtype: int64

In [16]:
# Verify row counts after funding-type filter — Series A
ai_series_a.count()

Transaction Name               683
Transaction Name URL           468
Organization Location          683
Organization Name              683
Organization Name URL          468
Funding Type                   683
Money Raised                   683
Money Raised Currency          468
Money Raised (in USD)          468
Announced Date                 683
Diversity Spotlight            328
Number of Partner Investors    540
Number of Investors            675
Investor Names                 675
dtype: int64

In [17]:
# Parse announcement dates to datetime for time-based operations
ai_seed["Announced Date"] = pd.to_datetime(ai_seed["Announced Date"])
ai_series_a["Announced Date"] = pd.to_datetime(ai_series_a["Announced Date"])

### 2.1 Collapse to One Round per Startup

For startups with multiple Seed or Series A rounds, retain only the **earliest** round of each type. This ensures one observation per firm in the survival dataset.

In [18]:
# earliest seed round per startup
ai_seed_first = (
    ai_seed.sort_values("Announced Date")
    .groupby("Organization Name", as_index=False)
    .first()
)

# earliest series A round per startup
ai_series_a_first = (
    ai_series_a.sort_values("Announced Date")
    .groupby("Organization Name", as_index=False)
    .first()
)

### 2.2 Merge Seed and Series A — Define Event and Censoring

Left-join seed-funded firms to Series A records. Firms without a Series A match are **right-censored** at the last observed date in the dataset.

In [19]:
# Right-censoring date: the last announcement observed across both datasets
censor_date = max(ai_seed["Announced Date"].max(),
                   ai_series_a["Announced Date"].max())

In [20]:
# Left-join: keep all seed-funded firms, attach Series A date if it exists
surv = ai_seed_first.merge(
    ai_series_a_first[["Organization Name", "Announced Date"]],
    on="Organization Name",
    how="left",
    suffixes=("_seed", "_series_A")
)

surv.head(3)

,Organization Name,Transaction Name,Transaction Name URL,Organization Location,Organization Name URL,Funding Type,Money Raised,Money Raised Currency,Money Raised (in USD),Announced Date_seed,Diversity Spotlight,Number of Partner Investors,Number of Investors,Investor Names,Announced Date_series_A
0,10by10.io,Seed Round - 10by10.io,https://www.crunchbase.com/funding_round/10-by...,"San Francisco, California, United States, Nort...",https://www.crunchbase.com/organization/10-by-10,Seed,120000,USD,120000,2017-08-22,"Women Founded, Women Led",NaN,1.0,Y Combinator,NaT
1,2B0 (To Be Zero),Seed Round - 2B0 (To Be Zero),https://www.crunchbase.com/funding_round/2b0-s...,"Menlo Park, California, United States, North A...",https://www.crunchbase.com/organization/2b0,Seed,250000,USD,250000,2020-07-12,None,NaN,NaN,None,NaT
2,310.ai,Seed Round - 310.ai,https://www.crunchbase.com/funding_round/310-a...,"San Francisco, California, United States, Nort...",https://www.crunchbase.com/organization/310-ai,Seed,3700000,USD,3700000,2023-01-01,None,1.0,2.0,"AVV, Cota Capital",NaT


In [21]:
# Rename for clarity
surv = surv.rename(columns={
    "Announced Date_seed": "seed_date",
    "Announced Date_series_A": "series_a_date"
})
surv.columns

Index(['Organization Name', 'Transaction Name', 'Transaction Name URL',
       'Organization Location', 'Organization Name URL', 'Funding Type',
       'Money Raised', 'Money Raised Currency', 'Money Raised (in USD)',
       'seed_date', 'Diversity Spotlight', 'Number of Partner Investors',
       'Number of Investors', 'Investor Names', 'series_a_date'],
      dtype='object')

In [22]:
# Event indicator: 1 if series a exists, 0 otherwise
surv["event_series_a"] = surv["series_a_date"].notna().astype(int)
surv.head()

,Organization Name,Transaction Name,Transaction Name URL,Organization Location,Organization Name URL,Funding Type,Money Raised,Money Raised Currency,Money Raised (in USD),seed_date,Diversity Spotlight,Number of Partner Investors,Number of Investors,Investor Names,series_a_date,event_series_a
0,10by10.io,Seed Round - 10by10.io,https://www.crunchbase.com/funding_round/10-by...,"San Francisco, California, United States, Nort...",https://www.crunchbase.com/organization/10-by-10,Seed,120000,USD,120000,2017-08-22,"Women Founded, Women Led",NaN,1.0,Y Combinator,NaT,0
1,2B0 (To Be Zero),Seed Round - 2B0 (To Be Zero),https://www.crunchbase.com/funding_round/2b0-s...,"Menlo Park, California, United States, North A...",https://www.crunchbase.com/organization/2b0,Seed,250000,USD,250000,2020-07-12,None,NaN,NaN,None,NaT,0
2,310.ai,Seed Round - 310.ai,https://www.crunchbase.com/funding_round/310-a...,"San Francisco, California, United States, Nort...",https://www.crunchbase.com/organization/310-ai,Seed,3700000,USD,3700000,2023-01-01,None,1.0,2.0,"AVV, Cota Capital",NaT,0
3,3Analytics,Seed Round - 3Analytics,https://www.crunchbase.com/funding_round/3anal...,"Cupertino, California, United States, North Am...",https://www.crunchbase.com/organization/3analy...,Seed,474996,USD,474996,2021-02-25,None,NaN,NaN,None,NaT,0
4,3DLOOK,Seed Round - 3DLOOK,https://www.crunchbase.com/funding_round/3dloo...,"San Mateo, California, United States, North Am...",https://www.crunchbase.com/organization/3dlook,Seed,1000000,USD,1000000,2018-07-10,Women Founded,NaN,2.0,"500 Global, u.ventures",2021-03-16,1


In [23]:
# Date we stop following the firm (events or censoring)
surv["end_date"] = surv["series_a_date"].fillna(censor_date)

In [24]:
surv["end_date"]

0      2024-12-23
1      2024-12-23
2      2024-12-23
3      2024-12-23
4      2021-03-16
          ...    
1168   2024-12-23
1169   2024-12-23
1170   2024-12-23
1171   2017-10-01
1172   2024-12-23
Name: end_date, Length: 1173, dtype: datetime64[ns]

In [25]:
# Time to event / censor in days and months
surv["time_to_series_a_days"] = (surv["end_date"] - surv["seed_date"]).dt.days
surv["time_to_series_a_months"] = surv["time_to_series_a_days"] / 30.44

In [26]:
# Preview the merged survival dataset
surv.head()

,Organization Name,Transaction Name,Transaction Name URL,Organization Location,Organization Name URL,Funding Type,Money Raised,Money Raised Currency,Money Raised (in USD),seed_date,Diversity Spotlight,Number of Partner Investors,Number of Investors,Investor Names,series_a_date,event_series_a,end_date,time_to_series_a_days,time_to_series_a_months
0,10by10.io,Seed Round - 10by10.io,https://www.crunchbase.com/funding_round/10-by...,"San Francisco, California, United States, Nort...",https://www.crunchbase.com/organization/10-by-10,Seed,120000,USD,120000,2017-08-22,"Women Founded, Women Led",NaN,1.0,Y Combinator,NaT,0,2024-12-23,2680,88.042050
1,2B0 (To Be Zero),Seed Round - 2B0 (To Be Zero),https://www.crunchbase.com/funding_round/2b0-s...,"Menlo Park, California, United States, North A...",https://www.crunchbase.com/organization/2b0,Seed,250000,USD,250000,2020-07-12,None,NaN,NaN,None,NaT,0,2024-12-23,1625,53.383706
2,310.ai,Seed Round - 310.ai,https://www.crunchbase.com/funding_round/310-a...,"San Francisco, California, United States, Nort...",https://www.crunchbase.com/organization/310-ai,Seed,3700000,USD,3700000,2023-01-01,None,1.0,2.0,"AVV, Cota Capital",NaT,0,2024-12-23,722,23.718791
3,3Analytics,Seed Round - 3Analytics,https://www.crunchbase.com/funding_round/3anal...,"Cupertino, California, United States, North Am...",https://www.crunchbase.com/organization/3analy...,Seed,474996,USD,474996,2021-02-25,None,NaN,NaN,None,NaT,0,2024-12-23,1397,45.893561
4,3DLOOK,Seed Round - 3DLOOK,https://www.crunchbase.com/funding_round/3dloo...,"San Mateo, California, United States, North Am...",https://www.crunchbase.com/organization/3dlook,Seed,1000000,USD,1000000,2018-07-10,Women Founded,NaN,2.0,"500 Global, u.ventures",2021-03-16,1,2021-03-16,980,32.194481


In [27]:
# Sanity check for no negative times
bad = surv[surv["time_to_series_a_days"] < 0]
print(len(bad))
bad.head()

0


,Organization Name,Transaction Name,Transaction Name URL,Organization Location,Organization Name URL,Funding Type,Money Raised,Money Raised Currency,Money Raised (in USD),seed_date,Diversity Spotlight,Number of Partner Investors,Number of Investors,Investor Names,series_a_date,event_series_a,end_date,time_to_series_a_days,time_to_series_a_months


In [28]:
# Confirm dataset dimensions after date parsing
surv.count()

Organization Name              1173
Transaction Name               1173
Transaction Name URL           1173
Organization Location          1173
Organization Name URL          1173
Funding Type                   1173
Money Raised                   1173
Money Raised Currency          1173
Money Raised (in USD)          1173
seed_date                      1173
Diversity Spotlight             279
Number of Partner Investors     552
Number of Investors            1001
Investor Names                 1001
series_a_date                   409
event_series_a                 1173
end_date                       1173
time_to_series_a_days          1173
time_to_series_a_months        1173
dtype: int64

---

## Part 3 — Metro Regions, Covariates & Sector Label

### 3.1 Extract City and State from Organization Location

Parse the comma-delimited `Organization Location` field into separate `city` and `state` columns for metro-region mapping.

In [29]:
# Split the comma-delimited Organization Location into components
loc_parts = surv["Organization Location"].str.split(",", expand=True)
print(loc_parts.head())

               0            1               2               3
0  San Francisco   California   United States   North America
1     Menlo Park   California   United States   North America
2  San Francisco   California   United States   North America
3      Cupertino   California   United States   North America
4      San Mateo   California   United States   North America


In [30]:
# Number of unique location strings
print(len(loc_parts))

1173


In [31]:
# Assign parsed city and state to the survival dataframe
surv["city"] = loc_parts[0].str.strip()
surv["state"] = loc_parts[1].str.strip()
print(surv.head())

  Organization Name               Transaction Name  \
0         10by10.io         Seed Round - 10by10.io   
1  2B0 (To Be Zero)  Seed Round - 2B0 (To Be Zero)   
2            310.ai            Seed Round - 310.ai   
3        3Analytics        Seed Round - 3Analytics   
4            3DLOOK            Seed Round - 3DLOOK   

                                Transaction Name URL  \
0  https://www.crunchbase.com/funding_round/10-by...   
1  https://www.crunchbase.com/funding_round/2b0-s...   
2  https://www.crunchbase.com/funding_round/310-a...   
3  https://www.crunchbase.com/funding_round/3anal...   
4  https://www.crunchbase.com/funding_round/3dloo...   

                               Organization Location  \
0  San Francisco, California, United States, Nort...   
1  Menlo Park, California, United States, North A...   
2  San Francisco, California, United States, Nort...   
3  Cupertino, California, United States, North Am...   
4  San Mateo, California, United States, North Am...   

 

### 3.2 Build City-to-Metro Template

Extract the unique (city, state) pairs and export as a template CSV. This template is manually mapped to metro regions (SFBA, GNYA, LA, Miami, Other) and saved as the lookup file loaded below.

In [32]:
# Extract unique (city, state) pairs for metro-region mapping
ai_cities = surv[["city", "state"]].drop_duplicates().reset_index(drop=True)
ai_cities.head()

,city,state
0,San Francisco,California
1,Menlo Park,California
2,Cupertino,California
3,San Mateo,California
4,San Jose,California


In [33]:
# Total number of unique city-state combinations
len(ai_cities)

102

In [34]:
# Export blank template for manual metro-region assignment
ai_cities.to_csv(r"..\data\cleaned\ai_loc_map\ai_city_to_metro_template.csv", index=False)

### 3.3 Merge Metro Region Mapping

Join the manually-curated city-to-metro lookup back to the survival dataset. The merge uses both `city` and `state` to avoid collisions. The `validate="m:1"` flag ensures each city maps to exactly one metro region.

In [35]:
# Load the completed city-to-metro mapping (manually curated from the template above)
ai_cities_to_metro = pd.read_csv(r"..\data\cleaned\ai_loc_map\ai_city_to_metro.csv")

In [36]:
# sanity: no duplicate (city, state) pairs
dupes = ai_cities_to_metro[ai_cities_to_metro.duplicated(["city", "state"], keep=False)]
print("Duplicate (city+state in mapping:\n", dupes)

Duplicate (city+state in mapping:
 Empty DataFrame
Columns: [city, state, metro_region]
Index: []


In [37]:
# Left-join metro region mapping onto the survival dataset (many firms → one metro)
surv = surv.merge(
    ai_cities_to_metro,
    on=["city", "state"],
    how="left", 
    validate="m:1"
)

In [38]:
# Quick duplicate check on firms
dupe_firms = surv["Organization Name"].value_counts()
print(dupe_firms[dupe_firms > 1].count())

0


### 3.4 Rename Crunchbase Variables

Map raw Crunchbase column names to shorter, analysis-friendly names used in the hazard model.

In [39]:
# Rename Crunchbase's dollar-formatted column to a shorter analysis name
surv["seed_amount_usd"] = surv["Money Raised (in USD)"]

In [40]:
# Carry forward investor count fields under shorter names
surv["num_partner_investors"] = surv["Number of Partner Investors"]
surv["num_investors_total"] = surv["Number of Investors"]

### 3.5 Sector Label and Cohort Year

Add the sector identifier (`AI`) and extract the seed year for cohort fixed effects in the hazard model.

In [41]:
# Extract cohort year from seed date for fixed effects in the hazard model
surv["seed_year"] = surv["seed_date"].dt.year

# Label all observations in this dataset as AI (will be pooled with FinTech later)
surv["sector"] = "AI" 

In [42]:
# Verify all expected columns are present before final selection
surv.columns

Index(['Organization Name', 'Transaction Name', 'Transaction Name URL',
       'Organization Location', 'Organization Name URL', 'Funding Type',
       'Money Raised', 'Money Raised Currency', 'Money Raised (in USD)',
       'seed_date', 'Diversity Spotlight', 'Number of Partner Investors',
       'Number of Investors', 'Investor Names', 'series_a_date',
       'event_series_a', 'end_date', 'time_to_series_a_days',
       'time_to_series_a_months', 'city', 'state', 'metro_region',
       'seed_amount_usd', 'num_partner_investors', 'num_investors_total',
       'seed_year', 'sector'],
      dtype='object')

---

## Part 4 — Final Dataset

Select the analysis-relevant columns, rename `Organization Name` to `firm_id` for consistency across sector datasets, and export the final survival CSV.

In [43]:
# Select only the columns needed for the survival analysis and rename firm identifier
simple_surv = surv[[
    "Organization Name",
    "sector",
    "seed_date",
    "series_a_date",
    "end_date",
    "time_to_series_a_months",
    "event_series_a",
    "metro_region",
    "seed_amount_usd",
    "num_partner_investors",
    "num_investors_total",
    "seed_year"
]].copy()

simple_surv = simple_surv.rename(columns={
    "Organization Name" : "firm_id"
})

simple_surv.head()

,firm_id,sector,seed_date,series_a_date,end_date,time_to_series_a_months,event_series_a,metro_region,seed_amount_usd,num_partner_investors,num_investors_total,seed_year
0,10by10.io,AI,2017-08-22,NaT,2024-12-23,88.042050,0,SFBA,120000,NaN,1.0,2017
1,2B0 (To Be Zero),AI,2020-07-12,NaT,2024-12-23,53.383706,0,SFBA,250000,NaN,NaN,2020
2,310.ai,AI,2023-01-01,NaT,2024-12-23,23.718791,0,SFBA,3700000,1.0,2.0,2023
3,3Analytics,AI,2021-02-25,NaT,2024-12-23,45.893561,0,SFBA,474996,NaN,NaN,2021
4,3DLOOK,AI,2018-07-10,2021-03-16,2021-03-16,32.194481,1,SFBA,1000000,NaN,2.0,2018


In [44]:
# Export the final AI survival dataset
simple_surv.to_csv(r"..\data\cleaned\cleaned_ai\ai_simple_survival.csv", index=False)